In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parents[0]))

import pandas as pd

from config import PROCESSED_DATA_DIR

In [2]:
icustays_df = pd.read_parquet(
    PROCESSED_DATA_DIR / "icustays_clean.parquet",
    columns=["stay_id"]
)

labevents_df = pd.read_parquet(PROCESSED_DATA_DIR / "labevents_clean.parquet")

print(icustays_df.shape)
print(labevents_df.shape)
print(labevents_df["lab_feature"].value_counts())

(94444, 1)
(12140259, 12)
lab_feature
potassium               612066
sodium                  610052
chloride                598694
hematocrit              577489
bicarbonate             575624
creatinine              574204
anion_gap               573044
bun                     572800
glucose                 562815
magnesium               559508
ph                      546288
phosphate               531419
calcium                 527227
hemoglobin              524367
platelets               523630
wbc                     514737
po2                     508835
pco2                    508351
ptt                     391648
pt                      356553
inr                     356421
lactate                 316101
ast                     161413
bilirubin_total         159614
alt                     159393
alkaline_phosphatase    158544
albumin                  79422
Name: count, dtype: int64


In [3]:
feature_window_hours = 24

labevents_24h_df = labevents_df[
    labevents_df["hours_from_icu_admit"].between(0, feature_window_hours, inclusive="both")
].copy()

print(labevents_24h_df.shape)
print(labevents_24h_df["lab_feature"].value_counts())

(4165852, 12)
lab_feature
ph                      211918
hematocrit              210550
po2                     199916
pco2                    199704
potassium               191162
chloride                190442
sodium                  187381
platelets               186507
hemoglobin              186029
creatinine              184622
bun                     184112
bicarbonate             183766
wbc                     183167
anion_gap               182156
glucose                 173408
magnesium               169796
phosphate               161124
calcium                 159968
lactate                 144018
ptt                     140680
pt                      136267
inr                     136193
ast                      58337
alt                      57843
bilirubin_total          57750
alkaline_phosphatase     57414
albumin                  31622
Name: count, dtype: int64


In [4]:
summary_features_df = (
    labevents_24h_df
    .groupby(["stay_id", "lab_feature"])["valuenum"]
    .agg(["count", "min", "max", "mean", "median", "std", "first", "last"])
    .reset_index()
)

summary_features_df["trend"] = summary_features_df["last"] - summary_features_df["first"]

summary_features_df.head()

,stay_id,lab_feature,count,min,max,mean,median,std,first,last,trend
0,30000153,anion_gap,2,12.0,12.0,12.0,12.0,0.000000,12.0,12.0,0.0
1,30000153,bicarbonate,2,19.0,23.0,21.0,21.0,2.828427,19.0,23.0,4.0
2,30000153,bun,2,22.0,22.0,22.0,22.0,0.000000,22.0,22.0,0.0
3,30000153,calcium,2,7.4,8.0,7.7,7.7,0.424264,7.4,8.0,0.6
4,30000153,chloride,2,115.0,115.0,115.0,115.0,0.000000,115.0,115.0,0.0


In [5]:
lab_features_wide_df = summary_features_df.pivot(
    index="stay_id",
    columns="lab_feature"
)

lab_features_wide_df.columns = [
    f"{feature}_{stat}_24h"
    for stat, feature in lab_features_wide_df.columns
]

lab_features_wide_df = lab_features_wide_df.reset_index()

lab_features_wide_df.head()

,stay_id,albumin_count_24h,alkaline_phosphatase_count_24h,alt_count_24h,anion_gap_count_24h,ast_count_24h,bicarbonate_count_24h,bilirubin_total_count_24h,bun_count_24h,calcium_count_24h,...,pco2_trend_24h,ph_trend_24h,phosphate_trend_24h,platelets_trend_24h,po2_trend_24h,potassium_trend_24h,pt_trend_24h,ptt_trend_24h,sodium_trend_24h,wbc_trend_24h
0,30000153,NaN,NaN,NaN,2.0,NaN,2.0,NaN,2.0,2.0,...,-3.0,0.01,1.1,-11.0,-6.0,0.4,0.0,0.0,3.0,-1.8
1,30000213,NaN,1.0,1.0,3.0,1.0,3.0,1.0,3.0,3.0,...,-7.0,0.07,0.8,0.0,-7.0,-0.4,NaN,NaN,-1.0,0.0
2,30000484,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,...,0.0,0.00,0.0,0.0,0.0,-0.2,0.0,0.0,0.0,0.0
3,30000646,NaN,1.0,1.0,3.0,1.0,3.0,1.0,3.0,3.0,...,0.0,0.00,-1.2,-2.0,0.0,0.7,-0.2,-2.1,5.0,-0.6
4,30000831,NaN,2.0,2.0,3.0,2.0,3.0,2.0,3.0,3.0,...,-3.0,0.04,0.2,-26.0,-19.0,0.0,0.0,-2.6,7.0,-7.1


In [6]:
observed_features_df = (
    labevents_24h_df
    .groupby(["stay_id", "lab_feature"])
    .size()
    .unstack(fill_value=0)
    .gt(0)
    .astype(int)
)

observed_features_df.columns = [
    f"{feature}_observed_24h"
    for feature in observed_features_df.columns
]

observed_features_df = observed_features_df.reset_index()

observed_features_df.head()

,stay_id,albumin_observed_24h,alkaline_phosphatase_observed_24h,alt_observed_24h,anion_gap_observed_24h,ast_observed_24h,bicarbonate_observed_24h,bilirubin_total_observed_24h,bun_observed_24h,calcium_observed_24h,...,pco2_observed_24h,ph_observed_24h,phosphate_observed_24h,platelets_observed_24h,po2_observed_24h,potassium_observed_24h,pt_observed_24h,ptt_observed_24h,sodium_observed_24h,wbc_observed_24h
0,30000153,0,0,0,1,0,1,0,1,1,...,1,1,1,1,1,1,1,1,1,1
1,30000213,0,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,0,0,1,1
2,30000484,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
3,30000646,0,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
4,30000831,0,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [7]:
labevents_features_df = icustays_df.merge(
    lab_features_wide_df,
    on="stay_id",
    how="left"
)

labevents_features_df = labevents_features_df.merge(
    observed_features_df,
    on="stay_id",
    how="left"
)

observed_cols = [col for col in labevents_features_df.columns if col.endswith("_observed_24h")]
labevents_features_df[observed_cols] = labevents_features_df[observed_cols].fillna(0).astype(int)

print(labevents_features_df.shape)
print(labevents_features_df["stay_id"].duplicated().sum())
print(labevents_features_df[observed_cols].sum().sort_values(ascending=False))

(94444, 271)
0
sodium_observed_24h                  90192
chloride_observed_24h                90189
potassium_observed_24h               90165
creatinine_observed_24h              90149
bicarbonate_observed_24h             90124
bun_observed_24h                     90124
anion_gap_observed_24h               90042
glucose_observed_24h                 89822
hematocrit_observed_24h              89809
platelets_observed_24h               89530
wbc_observed_24h                     89518
hemoglobin_observed_24h              89514
magnesium_observed_24h               87349
phosphate_observed_24h               84429
calcium_observed_24h                 83792
pt_observed_24h                      75563
inr_observed_24h                     75546
ptt_observed_24h                     74899
ph_observed_24h                      52569
po2_observed_24h                     50701
pco2_observed_24h                    50691
lactate_observed_24h                 50399
ast_observed_24h                     41

In [8]:
labevents_features_df.to_parquet(
    PROCESSED_DATA_DIR / "labevents_features.parquet",
    index=False
)